# Magnetic nanoparticles
We here show how to analyse (a subset) of the data on $\alpha$Fe$_2$O$_3$ nanoparticles presented in J. Chem. Phys. 140, 044709 (2014).

$\alpha$Fe$_2$O$_3$ is an antiferromagnet with magnetic moments in the hexagonal $ab$ plane. Magnetic nanoparticles are small enough to be single domain, and all magnetic moments therefore move coherently. Because the nanoparticles are so small, the magnetization can spontanously flip. This is called superparamagnetism, and if the characteristic flipping time is given by $\tau$ then the neutron signal from the magnetic structure is given by a Lorentzian with width (half width at half maximum) of $\Gamma = \hbar/\tau$:

$$
I_\text{SPM} = \frac{A_\text{SPM}}{\pi}\frac{\Gamma}{\Gamma^2+\hbar^2 \omega^2}.
$$

The nanoparticles also exhibit quantized spin-wave excitations. We here observe only two ${\bf Q}=0$ excitations, labelled $\hbar \omega_+$ and $\hbar \omega_-$, corresponding to movement within the $ab$ plane and perpendicular to it, respectively. These excitations occur at the magnetic Bragg peaks, which are the (003) ($|Q|=1.37$ Å$^{-1}$) and (101) ($|Q|=1.51$  Å$^{-1}$) reflections. The signal from these excitations can be well described by damped harmonic oscillators:
$$
I_\pm = \frac{A_\pm}{\pi} \frac{2\gamma_\pm \hbar \omega_\pm^2}{(\hbar^2 \omega^2 - \omega_{\pm}^2)^2 + 4\gamma_\pm^2 \hbar^2 \omega^2}.
$$
Here, $\gamma_\pm$ is the intrinsic width of the excitation

We are here interested in determining the flipping time, $\tau$ and the energy of the two excitations, $\omega_\pm$.

Several experiments were carried out, and we here analyse only a subset of the data, obtained at IN5 at ILL in Grenoble, France. We have the measured intensity as function of energy for four different values of $Q$; at the two magnetic peaks and at two positions where only background is present. The background is complicated and will need to be interpolated based on those measurements. We have data at 1.5 K, where the dynamics are almost completely frozen, to determine the resolution function, and at 150 K where we want to fit the data.

The background is complicated because water gets adsorbed on the surface of the nanoparticles. Some of the water is motionless and gives rise to elastic scattering, modelled by a delta function. Part of the water is mobile and therefore gives rise to quasielastic scattering, which is modelled by a Lorentzian. In total, the scattering from water is given by
$$
I_W = A_W \delta(\omega) + \frac{A_\text{W}}{\pi}\frac{\Gamma_W}{\Gamma_W^2+\hbar^2 \omega^2}.
$$

Enough background, let's start looking at the data! We first import everything that we need.

In [ ]:

import numpy as np
import scipp as sc

from easydynamics.analysis.analysis import Analysis
from easydynamics.experiment import Experiment
from easydynamics.sample_model import ComponentCollection
from easydynamics.sample_model import DampedHarmonicOscillator
from easydynamics.sample_model import DeltaFunction
from easydynamics.sample_model import Gaussian
from easydynamics.sample_model import Lorentzian
from easydynamics.sample_model import Polynomial
from easydynamics.sample_model.background_model import BackgroundModel
from easydynamics.sample_model.instrument_model import InstrumentModel
from easydynamics.sample_model.resolution_model import ResolutionModel
from easydynamics.sample_model.sample_model import SampleModel

# Make the plots interactive
%matplotlib widget

We begin with the resolution. We create an `Experiment`, download the data using `pooch` and attach it to the `Experiment`.

In [ ]:
resolution_experiment = Experiment(display_name='Nanoparticles, 1.5 K', data='data/nano_1p5K.h5')

We now take a look at the data. It is helpful to turn on the log scale since any variations in the data are tiny.

In [ ]:
resolution_experiment.plot_data(slicer=True, keep='energy')

We will use this data to determine the resolution function. There are some small exciations away from the elastic line that we do not wish to deal with. We therefore remove those dataoints using scipp, and plot the result.

In [ ]:
E_min = -0.2 * sc.Unit('meV')
E_max = 0.2 * sc.Unit('meV')
resolution_experiment.data = resolution_experiment.data['energy', E_min:E_max]

resolution_experiment.plot_data(slicer=True, keep='energy')

The resolution function can to a good approximation be modelled as a Gaussian. In truth, it has small Lorentzian tails, but we leave it as an exercise for the interested reader to add this.

We define the `SampleModel` to be a `DeltaFunction`, since there is essentially only elastic scattering present. We also add a constant background using a `Polynomial`. 

As in Tutorial 1 we place everything in an `Analysis` object and plot the start guesses.

In [ ]:
delta_function = DeltaFunction(area=100)
res_sample_model = SampleModel(components=delta_function)

res_resolution_model = ResolutionModel()
res_components = ComponentCollection()
res_gauss = Gaussian(area=1, width=0.02)
res_gauss.area.fixed = True

res_components.append_component(res_gauss)
res_resolution_model.components = res_components

background_model = BackgroundModel()
polynomial = Polynomial(coefficients=[1.5])
polynomial.coefficients[0].min = 0.0
background_model.components = polynomial


res_instrument_model = InstrumentModel(
    resolution_model=res_resolution_model,
    background_model=background_model,
)

res_analysis = Analysis(
    experiment=resolution_experiment,
    sample_model=res_sample_model,
    instrument_model=res_instrument_model,
)

res_analysis.plot_data_and_model(keep='energy')

The start guess looks reasonable, so let us fit everything and check out the result. The fit is not perfect, but it's certainly good enough for our purposes, giving excellent agreement over several orders of magnitude. 

In [ ]:
res_analysis.fit()
res_analysis.plot_data_and_model(keep='energy')

With our resolution in hand, it's time to look at the data. As before, we download the data file using pooch and make an `Experiment`.

In [ ]:
experiment = Experiment(display_name='Nanoparticles, 150 K', data='data/nano_150K.h5')
experiment.plot_data(slicer=True, keep='energy')

The magnetic excitations are around 1.1 meV and 0.3 meV, so we cut the data to $\pm1.5$ meV and plot the result. At the two lower $Q$, there is mainly the background signal from water as described above, whereas the signal at higher $Q$ is much more complex.

In [ ]:
E_min = -1.5 * sc.Unit('meV')
E_max = 1.5 * sc.Unit('meV')
experiment.data = experiment.data['energy', E_min:E_max]
experiment.data.variances[~np.isfinite(experiment.data.values)] = 1.0
experiment.data.values[~np.isfinite(experiment.data.values)] = 0.0

experiment.plot_data(slicer=True, keep='energy')

By now you know the routine: we create a `SampleModel` with the desired components, a `BackgroundModel` and an `InstrumentModel` using the resolution determined before. We furthermore set the temperature of the sample to be 150 K. This turns on detailed balancing, which has a small but non-zero effect on our fit. As always, we check out the start guesses before we fit. 

In [ ]:
sample_model = SampleModel()
water_delta_function = DeltaFunction(display_name='Water delta function', area=100)
water_lorentzian = Lorentzian(display_name='Water Lorentzian', area=10, width=0.2)
sample_model.append_component(water_delta_function)
sample_model.append_component(water_lorentzian)
sample_model.temperature = 150


background_model = BackgroundModel()
polynomial = Polynomial(coefficients=[0.15])
polynomial.coefficients[0].min = 0.0
background_model.components = polynomial


instrument_model = InstrumentModel(background_model=background_model)


analysis = Analysis(
    experiment=experiment, sample_model=sample_model, instrument_model=instrument_model
)
analysis.instrument_model._resolution_model = res_analysis.instrument_model.resolution_model
analysis.instrument_model.resolution_model.fix_all_parameters()

analysis.plot_data_and_model(keep='energy')

We carry out the fit and inspect the result. It's a decent fit at low $Q$, and obviously terrible at the magnetic positions. This is expected; we're only modelling the background and not the full signal yet. The fit at low $Q$ could also be improved, but it is good enough for our purposes.

In [ ]:
analysis.fit()
analysis.plot_data_and_model(keep='energy')

Now we want to figure out what the background from water looks like at the magnetic positions. To do this, we inspect the parameters describing it and look for patterns. It looks like the area of the delta function, the area of the Lorentzian and the width of the Lorentzian are almost identical for the two non-magnetic positions. That is promising, as we can then fix these values at the magnetic positions.

In [ ]:
analysis.plot_parameters(names=['Water delta function area', 'Water Lorentzian area'])

In [ ]:
analysis.plot_parameters(names=['Water Lorentzian width'])

We calculate a simple average of the relevant parameters, and fix these in our `Analysis` object at all Q. We plot the resulting model and see that it indeed fits well at low $Q$. At higher $Q$ it obviosuly does not describe all the signal, since there is magnetic scattering there as well.

The `DeltaFunction` is the first component (index 0), and the Lorentzian is the second component (index 1). It will soon be possible to refer to components by name as well as index. It will also be made easier to fix parameters at multiple $Q$, but for now we do it manually.

In [ ]:
delta_0 = analysis.sample_model.get_component_collection(Q_index=0).components[0]
delta_1 = analysis.sample_model.get_component_collection(Q_index=1).components[0]
delta_area = (delta_0.area + delta_1.area) / 2


lorz_0 = analysis.sample_model.get_component_collection(Q_index=0).components[1]
lorz_1 = analysis.sample_model.get_component_collection(Q_index=1).components[1]
lorz_area = (lorz_0.area + lorz_1.area) / 2
lorz_width = (lorz_0.width + lorz_1.width) / 2


for Q_index in range(analysis.sample_model.Q.size):
    delta = analysis.sample_model.get_component_collection(Q_index=Q_index).components[0]
    delta.area = delta_area.value
    delta.area.fixed = True

    lorz = analysis.sample_model.get_component_collection(Q_index=Q_index).components[1]
    lorz.area = lorz_area.value
    lorz.width = lorz_width.value
    lorz.area.fixed = True
    lorz.width.fixed = True

analysis.plot_data_and_model(keep='energy')

We are now ready to fit the magnetic signal, fixing the background from water as described above. We create a new `Analysis` object and add the water components and the components describing the magnetic signal as described above.

In [ ]:
# Now make a new analysis with this sample model
mag_sample_model = SampleModel()
water_delta_function = DeltaFunction(display_name='Water delta function', area=100)
water_lorentzian = Lorentzian(display_name='Water Lorentzian', area=100, width=0.2)
mag_sample_model.append_component(water_delta_function)
mag_sample_model.append_component(water_lorentzian)

# Add all the magnetic components
DHO1 = DampedHarmonicOscillator(display_name='DHO1', area=5, center=0.35, width=0.2)
DHO2 = DampedHarmonicOscillator(display_name='DHO2', area=1, center=1.1, width=0.1)
mag_lorz = Lorentzian(display_name='Magnetic Lorentzian', area=30, width=0.01)
mag_sample_model.append_component(DHO1)
mag_sample_model.append_component(DHO2)
mag_sample_model.append_component(mag_lorz)

background_model = BackgroundModel()
polynomial = Polynomial(coefficients=[0.15])
background_model.components = polynomial

instrument_model = InstrumentModel(background_model=background_model)


# Create the analysis object
mag_analysis = Analysis(
    experiment=experiment, sample_model=mag_sample_model, instrument_model=instrument_model
)
mag_analysis.instrument_model._resolution_model = res_analysis.instrument_model.resolution_model
mag_analysis.instrument_model.resolution_model.fix_all_parameters()

We now fix all the non-magnetic parameters to the values we found in the previous fit.


In [ ]:
for Q_index in range(mag_analysis.sample_model.Q.size):
    delta = mag_analysis.sample_model.get_component_collection(Q_index=Q_index).components[0]
    delta.area = delta_area.value
    delta.area.fixed = True

    lorz = mag_analysis.sample_model.get_component_collection(Q_index=Q_index).components[1]
    lorz.area = lorz_area.value
    lorz.width = lorz_width.value
    lorz.area.fixed = True
    lorz.width.fixed = True

We also fix all the parameters describing the magnetic signal at low Q. We set the areas to 0 and the center and width to fixed values so that they will not appear in the fit. We also take a look at the model before the fit.

In [ ]:
for Q_index in [0, 1]:
    DHO1 = mag_analysis.sample_model.get_component_collection(Q_index=Q_index).components[2]
    DHO2 = mag_analysis.sample_model.get_component_collection(Q_index=Q_index).components[3]
    lorz = mag_analysis.sample_model.get_component_collection(Q_index=Q_index).components[4]

    DHO1.area = 0.0
    DHO1.center = 1.0
    DHO2.width = 0.1
    DHO1.fix_all_parameters()

    DHO2.area = 0.0
    DHO2.center = 1.0
    DHO2.width = 0.1
    DHO2.fix_all_parameters()

    lorz.area = 0.0
    lorz.width = 0.1
    lorz.fix_all_parameters()

mag_analysis.plot_data_and_model(keep='energy')

It looks reasonable, so let's fit it and see what happens.

In [ ]:
mag_analysis.fit()
mag_analysis.plot_data_and_model(keep='energy')

The fit is very good, but we can make one improvement. The high-energy excitation is very weak at $Q=1.37$ Å$^{-1}$. We therefore make the width and center depend on the width and center of the same excitation at  $Q=1.51$ Å$^{-1}$

In [ ]:
DHO2_highQ = mag_analysis.sample_model.get_component_collection(Q_index=3).components[3]
DHO2_lowQ = mag_analysis.sample_model.get_component_collection(Q_index=2).components[3]

DHO2_lowQ.width.make_dependent_on('a', {'a': DHO2_highQ.width})
DHO2_lowQ.center.make_dependent_on('a', {'a': DHO2_highQ.center})

mag_analysis.plot_data_and_model(keep='energy')

In [ ]:
mag_analysis.fit(fit_method='simultaneous')
mag_analysis.plot_data_and_model(keep='energy')

In [ ]:
mag_analysis.plot_parameters(names=['DHO1 area', 'DHO2 area', 'Magnetic Lorentzian area'])

In [ ]:
mag_analysis.plot_parameters(names=['DHO1 width', 'DHO2 width', 'Magnetic Lorentzian width'])

In [ ]:
mag_analysis.plot_parameters(names=['DHO1 center', 'DHO2 center'])

In [ ]:
mag_analysis.sample_model.get_all_variables(Q_index=2)

In [ ]:
mag_analysis.sample_model.get_all_variables(Q_index=3)